In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer

In [7]:
chinese_c4 = load_dataset(
    "/mnt/sdb1/lijiaxin/Chinese-c4",
    split="train[:10%]"
)
chinese_c4

Dataset({
    features: ['url', 'timestamp', 'content_language', 'content_type', 'text'],
    num_rows: 1927301
})

In [8]:
tokenizer = AutoTokenizer.from_pretrained("/home/jx/experiment/BedRock-llm/tokenizer/tokenizer_modify")
tokenizer

TokenizersBackend(name_or_path='/home/jx/experiment/BedRock-llm/tokenizer/tokenizer_modify', vocab_size=6400, model_max_length=32768, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|im_start|>', 'eos_token': '<|im_end|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	6400: AddedToken("system", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	6401: AddedToken("user", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	6402: AddedToken("assistant", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [9]:
chinese_c4 = chinese_c4.train_test_split(
    test_size=0.1,
    seed=42,
    shuffle=True
)
chinese_c4['val'] = chinese_c4.pop('test')
chinese_c4

DatasetDict({
    train: Dataset({
        features: ['url', 'timestamp', 'content_language', 'content_type', 'text'],
        num_rows: 1734570
    })
    val: Dataset({
        features: ['url', 'timestamp', 'content_language', 'content_type', 'text'],
        num_rows: 192731
    })
})

In [10]:
result = tokenizer(chinese_c4['train'][3]["text"])
len(result["input_ids"])

669

In [11]:
# 超参数
BLOCK_SIZE = 1024

def tokenize_function(sample):
    outputs = tokenizer(
        sample["text"],
        truncation=True,
        max_length=BLOCK_SIZE,
        padding=False,

    )
    return outputs
        


In [12]:
result = tokenize_function(chinese_c4['train'][2])
print(result)

{'input_ids': [24, 1749, 4925, 1395, 270, 3172, 1331, 1075, 346, 97, 396, 242, 550, 110, 2486, 5074, 735, 1335, 1600, 3929, 976, 1676, 3818, 1064, 4998, 611, 1033, 5849, 368, 4588, 300, 108, 953, 270, 3650, 3172, 1331, 1075, 1733, 1024, 3345, 1492, 3044, 1025, 627, 829, 458, 117, 468, 107, 1520, 270, 671, 474, 111, 1075, 1733, 627, 829, 458, 117, 468, 107, 1404, 631, 2063, 623, 315, 1404, 631, 1612, 270, 1075, 866, 113, 572, 3483, 627, 829, 1086, 458, 117, 1520, 270, 1550, 631, 1761, 438, 105, 4162, 504, 244, 967, 6283, 468, 107, 4593, 3244, 518, 687, 315, 346, 97, 396, 242, 550, 110, 2486, 2606, 385, 245, 1468, 286, 3172, 1331, 1075, 308, 251, 1335, 2170, 2109, 341, 5074, 735, 1335, 1335, 931, 364, 250, 1452, 4564, 557, 394, 258, 5849, 581, 2910, 1531, 270, 3172, 1331, 1075, 1075, 931, 341, 5074, 735, 1335, 1335, 931, 2827, 4006, 5711, 922, 874, 5849, 581, 548, 3650, 286, 201, 364, 250, 1452, 4564, 2105, 3298, 3172, 1331, 1075, 308, 251, 1335, 315, 1075, 5074, 5769, 644, 270, 1325, 10

In [13]:
from torch.utils.data import Dataset, DataLoader


class PretrainDataset(Dataset):
    def __init__(self, dataset: Dataset, tokenizer, max_length=1024):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.dataset = dataset

    def tokenize_function(self, sample):
        outputs = tokenizer(
            sample["text"],
            truncation=True,
            max_length=self.max_length,
            padding=False
        )
        return outputs

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        return self.tokenize_function(self.dataset[index])

In [14]:
train_dataset = PretrainDataset(chinese_c4['train'], tokenizer, BLOCK_SIZE)
collate_fn = PretrainDataCollator()
dataloadr = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
for batch in dataloadr:
    print("input_ids:\n", batch["input_ids"])
    print("labels:\n", batch["labels"])
    print("attention_mask:\n", batch["attention_mask"])
    break

input_ids:
 tensor([[   1, 1083,    4,  ...,  518, 3490,    2],
        [   1, 1600, 3237,  ..., 1605,  902,    2],
        [   1, 1422, 1083,  ...,    0,    0,    0],
        ...,
        [   1,   42,   49,  ...,  165,  107,    2],
        [   1, 1380,  446,  ...,    0,    0,    0],
        [   1, 6397, 6157,  ..., 1486, 3032,    2]])
labels:
 tensor([[   1, 1083,    4,  ...,  518, 3490,    2],
        [   1, 1600, 3237,  ..., 1605,  902,    2],
        [   1, 1422, 1083,  ...,    0,    0,    0],
        ...,
        [   1,   42,   49,  ...,  165,  107,    2],
        [   1, 1380,  446,  ...,    0,    0,    0],
        [   1, 6397, 6157,  ..., 1486, 3032,    2]])
attention_mask:
 tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1]])


In [1]:
import torch
from transformers.data.data_collator import DataCollatorMixin

class PretrainDataCollator(DataCollatorMixin):
    def __init__(
        self,
        pad_token_id: int | None = 0,
        end_token_id: int | None = 2,
        bos_token_id: int | None = 1,
        ignore_index: int = -100,
        return_tensors: str = "pt"
    ):
        self.pad_token_id = pad_token_id
        self.end_token_id = end_token_id
        self.bos_token_id = bos_token_id
        self.ignore_index = ignore_index
        self.return_tensors = return_tensors

    def torch_call(self, examples: list[dict[str, any]]):
        input_ids_list = []

        # 1️⃣ 拼接 BOS/EOS
        for ex in examples:
            ids = ex["input_ids"]
            if isinstance(ids, list):
                ids = torch.tensor(ids, dtype=torch.long)  # 转成 tensor
            ids = torch.cat([
                torch.tensor([self.bos_token_id], dtype=torch.long),
                ids,
                torch.tensor([self.end_token_id], dtype=torch.long)
            ])
            input_ids_list.append(ids)

        # 2️⃣ 找到 batch 内最大长度
        max_len = max(len(ids) for ids in input_ids_list)

        batch_input_ids = []
        batch_attention_mask = []

        # 3️⃣ pad 
        for ids in input_ids_list:
            seq_len = len(ids)
            pad_len = max_len - seq_len

            # input_ids pad
            padded_input_ids = torch.cat([
                ids,
                torch.full((pad_len,), self.pad_token_id, dtype=torch.long)
            ])
            batch_input_ids.append(padded_input_ids)

            # attention mask
            attention_mask = (padded_input_ids != self.pad_token_id).long()
            batch_attention_mask.append(attention_mask)

        batch = {
            "input_ids": torch.stack(batch_input_ids),
            "labels": torch.stack(batch_input_ids),
            "attention_mask": torch.stack(batch_attention_mask),
        }

        return batch


/home/jx/miniconda3/envs/qwen3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
